# 02 — 长期记忆（Long-Term Memory）

**来源:** [LangChain Docs — Memory](https://docs.langchain.com/oss/python/deepagents/memory)

Memory 让 Agent 跨对话学习与改进。Deep Agents 将记忆实现为文件系统：Agent 通过文件读写记忆，你通过 Backend 控制存储位置。本 Notebook 覆盖**长期记忆**（跨对话持久化）。短期记忆（对话历史和临时文件）由系统自动管理。

## 记忆工作方式

| 阶段 | 行为 |
|------|------|
| 启动时注入 | Agent 将 memory 文件加载到 system prompt 中 |
| 运行时按需读取 | Skills 使用按需加载，只在匹配任务时读取完整内容 |
| 更新记忆 | Agent 通过 `edit_file` 工具更新记忆文件 |
| 持久化 | 变更在下一轮对话中可用 |

两种最常见的作用域模式：
- **Agent 作用域** — 所有用户共享同一份记忆
- **用户作用域** — 每个用户有独立的记忆文件

## 1. Agent-Scoped Memory（Agent 作用域记忆）

Agent 拥有自己的持久化身份。所有用户共享同一份记忆，Agent 通过每次对话积累知识、完善人格。关键点：Backend 的 `namespace` 设为 `(assistant_id,)`。

In [ ]:
# Agent 作用域记忆：所有用户共享同一份记忆文件
# 注意：访问 rt.server_info 需要 deepagents>=0.5.0

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from dotenv import load_dotenv
load_dotenv()

agent = create_deep_agent(
    model="deepseek-v4-flash",
    memory=["/memories/AGENTS.md"],
    skills=["/skills/"],
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            "/memories/": StoreBackend(
                namespace=lambda rt: (rt.server_info.assistant_id,),  # Agent 作用域
            ),
            "/skills/": StoreBackend(
                namespace=lambda rt: (rt.server_info.assistant_id,),
            ),
        },
    ),
)


### 1.1 完整示例：播种记忆 + 跨对话调用

先在 Store 中写入初始记忆，然后通过两个线程调用 Agent，验证 Agent 能记住并更新信息。

In [ ]:
# 完整示例：播种记忆并跨对话调用 Agent

from langchain_core.utils.uuid import uuid7

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from deepagents.backends.utils import create_file_data
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()  # 部署到 LangSmith 时应使用平台 Store

# 播种初始记忆文件
store.put(
    ("my-agent",),
    "/memories/AGENTS.md",
    create_file_data("""## Response style
- Keep responses concise
- Use code examples where possible
"""),
)

# 播种一个 skill
store.put(
    ("my-agent",),
    "/skills/langgraph-docs/SKILL.md",
    create_file_data("""---
name: langgraph-docs
description: Fetch relevant LangGraph documentation to provide accurate guidance.
---

# langgraph-docs

Use the fetch_url tool to read https://docs.langchain.com/llms.txt, then fetch relevant pages.
"""),
)

agent = create_deep_agent(
    model="deepseek-v4-flash",
    memory=["/memories/AGENTS.md"],
    skills=["/skills/"],
    backend=lambda rt: CompositeBackend(
        default=StateBackend(rt),
        routes={
            "/memories/": StoreBackend(
                rt, namespace=lambda rt: ("my-agent",)
            ),
            "/skills/": StoreBackend(
                rt, namespace=lambda rt: ("my-agent",)
            ),
        },
    ),
    store=store,
)

# 线程 1：告诉 Agent 一个新的偏好，它会保存到记忆
config1 = {"configurable": {"thread_id": str(uuid7())}}
agent.invoke(
    {"messages": [{"role": "user", "content": "I prefer detailed explanations. Remember that."}]},
    config=config1,
)

# 线程 2：Agent 读取记忆，应用偏好
config2 = {"configurable": {"thread_id": str(uuid7())}}
agent.invoke(
    {"messages": [{"role": "user", "content": "Explain how transformers work."}]},
    config=config2,
)

## 2. User-Scoped Memory（用户作用域记忆）

每个用户拥有独立的记忆文件。Agent 记住每个用户的偏好，核心 Agent 指令保持不变。Namespace 使用 `(user_id,)` 实现隔离。

In [ ]:
# 用户作用域记忆：每个用户有独立的记忆文件

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

agent = create_deep_agent(
    model="deepseek-v4-flash",
    memory=["/memories/preferences.md"],
    skills=["/skills/"],
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            "/memories/": StoreBackend(
                namespace=lambda rt: (rt.server_info.user.identity,),  # 用户作用域
            ),
            "/skills/": StoreBackend(
                namespace=lambda rt: (rt.server_info.user.identity,),
            ),
        },
    ),
)

### 2.1 完整示例：用户隔离记忆

播种不同用户的偏好记忆，两个用户各自只能看到自己的偏好。

In [ ]:
# 完整示例：播种每个用户的记忆，验证隔离

from langchain_core.utils.uuid import uuid7

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from deepagents.backends.utils import create_file_data
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()  # 部署到 LangSmith 时应使用平台 Store

# 为两个用户分别播种不同的偏好
store.put(
    ("user-alice",),
    "/memories/preferences.md",
    create_file_data("""## Preferences
- Likes concise bullet points
- Prefers Python examples
"""),
)
store.put(
    ("user-bob",),
    "/memories/preferences.md",
    create_file_data("""## Preferences
- Likes detailed explanations
- Prefers TypeScript examples
"""),
)

# 为 Alice 播种一个 skill
store.put(
    ("user-alice",),
    "/skills/langgraph-docs/SKILL.md",
    create_file_data("""---
name: langgraph-docs
description: Fetch relevant LangGraph documentation to provide accurate guidance.
---

# langgraph-docs

Use the fetch_url tool to read https://docs.langchain.com/llms.txt, then fetch relevant pages.
"""),
)

agent = create_deep_agent(
    model="deepseek-v4-flash",
    memory=["/memories/preferences.md"],
    skills=["/skills/"],
    backend=lambda rt: CompositeBackend(
        default=StateBackend(rt),
        routes={
            "/memories/": StoreBackend(
                rt,
                namespace=lambda rt: (rt.server_info.user.identity,),
            ),
            "/skills/": StoreBackend(
                rt,
                namespace=lambda rt: (rt.server_info.user.identity,),
            ),
        },
    ),
    store=store,
)

# 部署后，每个认证请求的 rt.server_info.user.identity 自动解析为相应用户
# Alice 和 Bob 各自只能看到自己的偏好
agent.invoke(
    {"messages": [{"role": "user", "content": "How do I read a CSV file?"}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)

## 3. 高级用法参数一览

| 维度 | 问题 | 选项 |
|------|------|------|
| 持续时间 | 记忆持续多久？ | 短期（单对话） vs 长期（跨对话） |
| 信息类型 | 存储什么类型的信息？ | 情景（经历）、程序性（技能）、语义（事实） |
| 作用域 | 谁可以看到和修改？ | 用户、Agent、组织 |
| 更新策略 | 何时写入记忆？ | 对话中（默认） vs 对话间（后台合并） |
| 检索方式 | 如何读取记忆？ | 注入 prompt（默认） vs 按需加载（skills） |
| Agent 权限 | Agent 能否写入？ | 读写（默认） vs 只读（共享策略） |

## 4. 情景记忆（Episodic Memory）

情景记忆保存完整的对话上下文。Deep Agents 使用 Checkpointer 实现：每次对话被持久化为一个带检查点的线程。要使历史对话可搜索，将线程搜索封装为一个工具。

In [ ]:
# 情景记忆：搜索历史对话的工具
# 将历史对话的搜索封装为工具，供 Agent 按需检索

from langgraph_sdk import get_client
from langchain.tools import tool, ToolRuntime

client = get_client(url="<DEPLOYMENT_URL>")

@tool
async def search_past_conversations(query: str, runtime: ToolRuntime) -> str:
    """搜索历史对话以获取相关上下文"""
    user_id = runtime.server_info.user.identity  # 从运行时上下文获取用户 ID
    threads = await client.threads.search(
        metadata={"user_id": user_id},
        limit=5,
    )
    results = []
    for thread in threads:
        history = await client.threads.get_history(thread_id=thread["thread_id"])
        results.append(history)
    return str(results)

In [ ]:
# 按用户或组织范围搜索对话

# 搜索特定用户的对话
threads = await client.threads.search(
    metadata={"user_id": user_id},
    limit=5,
)

# 搜索组织范围的对话
threads = await client.threads.search(
    metadata={"org_id": org_id},
    limit=5,
)

## 5. 组织级记忆（Organization-Level Memory）

使用组织范围的 namespace 替代用户级别。适用于跨所有用户和 Agent 的策略或知识。通常设置为**只读**以防止通过共享状态进行提示注入。

In [ ]:
# 组织级记忆：所有用户共享组织策略

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

agent = create_deep_agent(
    model="deepseek-v4-flash",
    memory=[
        "/memories/preferences.md",
        "/policies/compliance.md",   # 组织级策略文件
    ],
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            "/memories/": StoreBackend(
                namespace=lambda rt: (rt.server_info.user.identity,),
            ),
            "/policies/": StoreBackend(
                namespace=lambda rt: (rt.context.org_id,),  # 组织作用域
            ),
        },
    ),
)

In [ ]:
# 从应用代码播种组织级记忆

from langgraph_sdk import get_client
from deepagents.backends.utils import create_file_data

client = get_client(url="<DEPLOYMENT_URL>")

await client.store.put_item(
    (org_id,),
    "/compliance.md",
    create_file_data("""## Compliance policies
- Never disclose internal pricing
- Always include disclaimers on financial advice
"""),
)

## 6. 后台合并（Background Consolidation）

默认 Agent 在对话过程中（hot path）写入记忆。替代方案是在对话之间作为后台任务处理记忆，称为 **sleep time compute**。

| 方式 | 优点 | 缺点 |
|------|------|------|
| Hot path（对话中） | 立即可用，对用户透明 | 增加延迟，Agent 需多任务 |
| 后台合并（对话间） | 无用户感知延迟，可跨多对话综合 | 下次对话才可用，需额外 Agent |

### 6.1 合并 Agent

读取最近的对话历史，将关键事实合并到记忆存储中。

In [ ]:
# 合并 Agent：读取近期对话，合并关键事实到记忆存储

from datetime import datetime, timedelta, timezone

from deepagents import create_deep_agent
from langchain.tools import tool, ToolRuntime
from langgraph_sdk import get_client

sdk_client = get_client(url="<DEPLOYMENT_URL>")


@tool
async def search_recent_conversations(query: str, runtime: ToolRuntime) -> str:
    """搜索此用户在最近 6 小时内更新的对话"""
    user_id = runtime.server_info.user.identity

    since = datetime.now(timezone.utc) - timedelta(hours=6)
    threads = await sdk_client.threads.search(
        metadata={"user_id": user_id},
        updated_after=since.isoformat(),
        limit=20,
    )
    conversations = []
    for thread in threads:
        history = await sdk_client.threads.get_history(
            thread_id=thread["thread_id"]
        )
        conversations.append(history["values"]["messages"])
    return str(conversations)


agent = create_deep_agent(
    model="deepseek-v4-flash",
    system_prompt="""Review recent conversations and update the user's memory file.
Merge new facts, remove outdated information, and keep it concise.""",
    tools=[search_recent_conversations],
)

### 6.2 注册合并 Agent 到 langgraph.json

将合并 Agent 与主 Agent 一起注册到部署配置中。

In [ ]:
# langgraph.json 配置：注册主 Agent 和合并 Agent

config = {
  "dependencies": ["."],
  "graphs": {
    "agent": "./agent.py:agent",
    "consolidation_agent": "./consolidation_agent.py:agent"
  },
  "env": ".env"
}
print(config)

In [ ]:
# 使用 Cron 定时调度合并 Agent

from langgraph_sdk import get_client

client = get_client(url="<DEPLOYMENT_URL>")

cron_job = await client.crons.create(
    assistant_id="consolidation_agent",
    schedule="0 */6 * * *",          # 每 6 小时运行一次
    input={"messages": [{"role": "user", "content": "Consolidate recent memories."}]},
)

# Cron 间隔必须与合并 Agent 内 search_recent_conversations 的回看窗口匹配
# 这里都是 6 小时

## 7. 只读 vs 可写记忆

| 权限 | 使用场景 | 工作原理 |
|------|----------|----------|
| 读写（默认） | 用户偏好、Agent 自我改进、学到的技能 | Agent 通过 `edit_file` 工具更新 |
| 只读 | 组织策略、合规规则、共享知识库、开发者定义的技能 | 通过应用代码或 Store API 填充，使用 Permissions 拒绝写入 |

安全考虑：如果一个用户能写入其他用户读取的记忆，恶意用户可能向共享状态注入指令。应对措施：默认使用用户作用域；共享策略使用只读记忆；关键写入增加人工审批。

## 8. 并发写入 & 多 Agent

### 并发写入
多个线程并发写入同一文件可能导致 last-write-wins 冲突。对于用户作用域记忆很少见（用户通常一次一个对话）。对于 Agent 或组织作用域，考虑使用后台合并来序列化写入，或将记忆结构化为每个主题独立文件。

### 多个 Agent 同一部署
要给每个 Agent 独立的记忆，在 namespace 中加入 `assistant_id`：

In [ ]:
# 多 Agent 隔离：在 namespace 中加入 assistant_id

StoreBackend(
    namespace=lambda rt: (
        rt.server_info.assistant_id,
        rt.server_info.user.identity,
    ),
)

# 如果只需要 Agent 级别隔离，不需要用户级别，单独使用 assistant_id 即可

---
**参考链接**

- [Memory — LangChain Docs](https://docs.langchain.com/oss/python/deepagents/memory)
- [Documentation Index](https://docs.langchain.com/llms.txt)
- [Deep Agents Customization](https://docs.langchain.com/oss/python/deepagents/customization)